# Fix bugs in data merging of ESCO and O*NET
Felix Zaussinger | 13.04.2022

## Core Analysis Goal(s)
1. Fix bugs in occupation and skills metadata merging.
2. Gain confidence that the final datasets are valid.
3.

## Key Insight(s)
1.
2.
3.

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
# import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

KeyboardInterrupt: 

In [ ]:
from src import utils, stats_utils, plotting_utils
from src.data.framework import Esco, Onet, Crosswalks

Code ...

In [ ]:
esco = Esco()
crosswalks = Crosswalks()
onet = Onet()

### Checking problems with ONET-ESCO crosswalk
- some ONET greening occupations missing in Nesta crosswalk

In [ ]:
gt_esco = esco.read_greenness_task_based()

In [ ]:
gt_esco.title_gtp.unique()

63 ONET greening occupations dont have a match to ESCO based on current crosswalk

In [ ]:
bsel = gt_esco.loc[:, ["preferred_label"]].isna().values
df_no_match = gt_esco.loc[bsel]
df_no_match

In [ ]:
gt_esco.columns

Summary: number of ESCO occupations matched to a single greening ONET occupation

In [ ]:
grouping_cols = ["onet_code", "title_gtp", "occupation_type", "share_green_gtp"]

match_summary = gt_esco.groupby(grouping_cols)["preferred_label"].count().sort_values(ascending=False)
match_summary.rename("n_matches_esco", inplace=True)

match_summary.to_csv(
    os.path.join(useful_paths.results_dir, "validation", "onet_esco_crosswalk", "onet_to_esco_match_summary.csv"),
    sep=";"
)

match_summary

In [ ]:
missing_matches = match_summary[match_summary == 0].reset_index()
missing_matches

63 occupations are completely missing any match to ESCO occupations. What is going on there?

In [ ]:
missing_matches.shape

Inspecting the crosswalk I used for the thesis

In [ ]:
cw = crosswalks.onet_esco_mcc_full
cw

2942 ESCO occupations are matched to 669 ONET occupations

In [ ]:
cw.onet_code.unique().shape

None of these 63 onet occupations is part of this crosswalk

In [ ]:
for code, label in zip(missing_matches.onet_code, missing_matches.title_gtp):
    match_original_code = code in cw.onet_code.values
    print("{} ¦ {} ¦ {}".format(code, label, match_original_code))

#### Option 1: assign value to coarser SOC level and check if there is a match

In [ ]:
cw.query("onet_code == '51-4021.00'")
cw_unique_onet = cw.drop_duplicates(subset="onet_code")

sel = cw_unique_onet.onet_code == "11-9199.00"
cw_unique_onet.loc[sel, 'onet_occupation'].values.tolist()[0]

In [ ]:
summary_dict = {
    "onet_code_orig": [],
    "onet_label_orig": [],
    "onet_match_orig": [],
    "onet_code_1_below": [],
    "onet_label_1_below": [],
    "onet_match_1_below": [],
    "onet_code_2_below": [],
    "onet_label_2_below": [],
    "onet_match_2_below": []
}

for code_orig, label_orig in zip(missing_matches.onet_code, missing_matches.title_gtp):
    # coarser onet soc codes
    onet_code_1_below = code_orig[:-1] + "0"
    onet_code_2_below = code_orig[:-2] + "00"

    # original match with cw?
    match_orig = code_orig in cw_unique_onet.onet_code.values

    # if not, check one level below if not
    if match_orig is False:

        onet_match_1_below = onet_code_1_below in cw_unique_onet.onet_code.values
        if onet_match_1_below:
            sel = cw_unique_onet.onet_code == onet_code_1_below
            onet_label_1_below = cw_unique_onet.loc[sel, 'onet_occupation'].values.tolist()[0]

            onet_code_2_below = False
            onet_label_2_below = False
            onet_match_2_below = False
        else:
            onet_label_1_below = None

            # check if there is a match two levels below
            onet_match_2_below = onet_code_2_below in cw_unique_onet.onet_code.values
            if onet_match_2_below:
                sel_2 = cw_unique_onet.onet_code == onet_code_2_below
                onet_label_2_below = cw_unique_onet.loc[sel_2, 'onet_occupation'].values.tolist()[0]
                pass

    else:
        onet_match_1_below = None
        onet_label_1_below = None

    # update dict
    summary_dict["onet_code_orig"].append(code_orig)
    summary_dict["onet_label_orig"].append(label_orig)
    summary_dict["onet_match_orig"].append(match_orig)

    summary_dict["onet_code_1_below"].append(onet_code_1_below)
    summary_dict["onet_label_1_below"].append(onet_label_1_below)
    summary_dict["onet_match_1_below"].append(onet_match_1_below)

    summary_dict["onet_code_2_below"].append(onet_code_2_below)
    summary_dict["onet_label_2_below"].append(onet_label_2_below)
    summary_dict["onet_match_2_below"].append(onet_match_2_below)


df_cw_coarsening_summary = pd.DataFrame.from_dict(summary_dict)

df_cw_coarsening_summary

Results of downwalking
- 24 additional matches through downwalking to SOC 6D level
- 22 additional matches one level below (SOC 7D), 2 matches two levels below (SOC 8D)

In [ ]:
df_cw_coarsening_summary.onet_match_1_below.sum()

In [ ]:
df_cw_coarsening_summary.onet_match_2_below.sum()

#### Option 2: use reduced crosswalk at ESCO LVL 5
Idea
- use reduced crosswalk file used in MCC report that matches at the ESCO lvl 5.
- Assign same membership to all lower-level children.

Conclusion
- no additional matches are found, the crosswalk would need to be reconstructed.

In [ ]:
esco5_onet_cw = crosswalks.onet_esco_mcc_reduced
esco5_onet_cw.concept_uri.unique()

In [ ]:
matches =  []
for code_orig, label_orig in zip(missing_matches.onet_code, missing_matches.title_gtp):
    match = esco5_onet_cw.loc[esco5_onet_cw.loc[:, "onet_code"] == code_orig]
    matches.append(not match.empty)
    print(code_orig, "¦", label_orig, "¦", not match.empty)

In [ ]:
np.array(matches).sum()

#### Option 3: use ONET-SOC and SOC-ISCO crosswalks from MCC project (for LFS data only)

Idea:
- don't go via ESCO but directly match the GTP data to ISCO occupations

Conclusions:
- 56/63 occupations can be matched in this way.
- Not attaching to ESCO, however, will make us loose a lot of information when working with the national (DE/IT) data.

In [ ]:
crosswalks.soc10_isco08_ibs

In [ ]:
onet_us2010soc = crosswalks.onet_soc10
onet_us2010soc

In [ ]:
onet_us2010soc.iloc[:, 2].unique()

How many matches do I find in the ONET-SOC crosswalk?

In [ ]:
matches =  []
for code_orig, label_orig in zip(missing_matches.onet_code, missing_matches.title_gtp):
    match = onet_us2010soc.loc[onet_us2010soc.loc[:, 'O*NET-SOC 2010 Code'] == code_orig]
    matches.append(not match.empty)
    print(code_orig, "¦", label_orig, "¦", not match.empty)

56 of the 63 matchless occupation can be matched!

In [ ]:
np.array(matches).sum()

More detailed analysis

#### Option 4: use ONET SOC to ISCO crosswalk from the Institute for Structural Research - IBS

Idea:
- use an existing crosswalk from ONET-SOC to ISCO
-

In [ ]:
soc10_isco08 = crosswalks.soc10_isco08_ibs
soc10_isco08

In [ ]:
soc10_isco08.soc10.unique()

## Green/brown occupation master file
- triangulation via tasks and skills

In [ ]:
smd = esco.combine_skills_metadata(variable_selection=None)
omd = esco.combine_occupation_metadata(skills_metadata=smd)

In [ ]:
omd

1 ONET: Vona 2018 (Greenness), SOC 8-digit matched via ONET-ESCO crosswalk [CHECK]
2 ONET: GTP 2011 (Greenness), SOC 8-digit matched via ONET-ESCO crosswalk [CHECK]
3 ONET: Vona 2019 (Greenness), SOC 6-digit matched via IBS SOC-ISCO crosswalk
4 ONET: JRC/Consoli (Greenness), CP-2011 5-digit matched via CP2011-ESCO crosswalk
5 ESCO: ESCO 2022 (Greenness) [CHECK]

In [ ]:
cols = ['conceptUri', 'preferredLabel',
       'isco_level_4', 'isco_level_1', 'isco_level_2', 'isco_level_3',
       'isco_label_1', 'isco_label_2', 'isco_label_3', 'isco_label_4',
       'share_green_esco', 'share_brown_esco', 'gbn_classification_esco',
       'share_green_esco_ess', 'share_brown_esco_ess', 'gbn_classification_esco_ess',
       'onet_code', 'title_gtp', 'share_green_gtp', 'share_green_vona2018',
       'is_brown_onet', 'is_green_onet', 'gbn_classification_onet']

In [ ]:
omd_sub = omd[cols]

Fill nan values in ONET greenness shares with zeros

In [ ]:
omd_sub.loc[:, "share_green_gtp"] = omd_sub.share_green_gtp.fillna(0)
omd_sub.loc[:, "share_green_vona2018"] = omd_sub.share_green_vona2018.fillna(0)

**Triangulation of green occupations based on intersecting task- and skill data**

Distribution of shares, descriptive statistics

In [ ]:
omd_sub.describe()

In [ ]:
omd_sub.hist(bins=20, log=False, sharex=True)
plt.tight_layout()

Distributions quite skewed, use median

In [ ]:
omd_sub.select_dtypes("float").apply(stats_utils.naniqr, axis=0)

In [ ]:
omd_sub.select_dtypes("float").quantile(0.90)

In [ ]:
omd_sub.loc[omd_sub.loc[:, "share_green_esco"] > 0.2]

In [ ]:
omd_sub.select_dtypes("float")

In [ ]:
omd_sub[(omd_sub.share_green_esco_ess > 0) & ((omd_sub.share_green_gtp > 0) | (omd_sub.share_green_vona2018 > 0))]

In [ ]:
omd_sub[(omd_sub.share_brown_esco > 0) & omd_sub.is_brown_onet]